# Tutorial: Colab GPU Repo Smoke Test

Audience:
- You want to open this repository in VS Code or Google Colab and verify that the core code paths work before launching a long training run.

Prerequisites:
- In a clean Colab runtime, clone the repository first.
- In VS Code, open the notebook from inside the cloned repository.

Learning goals:
- Get the repository from GitHub.
- Verify that Python can import the repo.
- Verify that GPU is visible to the runtime.
- Run a small synthetic benchmark through the real project functions.
- Optionally launch the real pipeline when the dataset paths are present.


## Outline

1. Clone the repository from GitHub
2. Install dependencies if needed
3. Check Python, paths, and GPU visibility
4. Import repo modules from `src/`
5. Run a synthetic benchmark smoke test
6. Optionally run the real pipeline with `configs/colab_gpu.json`


In [3]:
# Run this in a clean Colab runtime before anything else.
# If the repo is already cloned and this notebook is opened from that repo, skip this cell.

%cd /content
!rm -rf risk-case-championship
!git clone https://github.com/Tech-Alash/risk-case-championship.git
%cd /content/risk-case-championship


/content
Cloning into 'risk-case-championship'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 129 (delta 30), reused 126 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 9.58 MiB | 10.39 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/risk-case-championship


In [4]:
# Optional install cell for a fresh Colab runtime.
# Run this after cloning and changing into the repo directory.

%pip install -r requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 36.9 MB/s eta 0:00:00


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from __future__ import annotations

import platform
import subprocess
import sys
from pathlib import Path

SEED = 42
EXPECTED_REPO_NAME = "risk-case-championship"

REPO_DIR = Path.cwd().resolve()
for candidate in [REPO_DIR, *REPO_DIR.parents]:
    if (candidate / ".git").exists() and (candidate / "src").exists():
        REPO_DIR = candidate
        break

print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "cwd": str(Path.cwd()),
    "repo_dir": str(REPO_DIR),
    "repo_name_ok": REPO_DIR.name == EXPECTED_REPO_NAME,
})

if not (REPO_DIR / "src").exists():
    raise RuntimeError("Repository not found. Clone it first and run the notebook from inside the repo.")

try:
    gpu_info = subprocess.run(
        ["nvidia-smi"],
        check=False,
        capture_output=True,
        text=True,
    )
    HAS_GPU = gpu_info.returncode == 0
    print("HAS_GPU =", HAS_GPU)
    if HAS_GPU:
        print(gpu_info.stdout[:1200])
    else:
        print(gpu_info.stderr[:400] or "nvidia-smi not available")
except FileNotFoundError:
    HAS_GPU = False
    print("nvidia-smi not found")


{'python': '3.12.12', 'platform': 'Linux-6.6.113+-x86_64-with-glibc2.35', 'cwd': '/content/risk-case-championship', 'repo_dir': '/content/risk-case-championship', 'repo_name_ok': True}
HAS_GPU = True
Sun Mar 15 09:59:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             16

In [6]:
import json
import random

import numpy as np
import pandas as pd

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from risk_case.models.benchmark import BenchmarkConfig, run_model_benchmark

random.seed(SEED)
np.random.seed(SEED)

colab_cfg_path = REPO_DIR / "configs" / "colab_gpu.json"
if colab_cfg_path.exists():
    colab_cfg = json.loads(colab_cfg_path.read_text(encoding="utf-8"))
    print("Loaded config:", colab_cfg_path)
    print("Benchmark candidates:", colab_cfg["benchmark"]["candidates"])
else:
    print("Config not found:", colab_cfg_path)


Loaded config: /content/risk-case-championship/configs/colab_gpu.json
Benchmark candidates: ['baseline_freq_sev', 'xgboost_freq_sev', 'catboost_freq_sev', 'oof_blend_freq_sev']


## Step 1 - Synthetic benchmark smoke test

This cell uses the real project benchmark entry point on a tiny generated dataset.
It is meant to verify imports, candidate construction, OOF blend logic, and GPU-aware CatBoost config without needing the real training CSV.


In [7]:
rows = 180
rng = np.random.default_rng(SEED)
is_claim = (rng.random(rows) < 0.18).astype(int)
claim_amount = is_claim * rng.gamma(shape=2.0, scale=700.0, size=rows)

df = pd.DataFrame(
    {
        "contract_number": [f"c_{i}" for i in range(rows)],
        "premium": rng.uniform(2500.0, 15000.0, rows),
        "premium_wo_term": rng.uniform(2000.0, 14000.0, rows),
        "is_claim": is_claim,
        "claim_amount": claim_amount,
        "claim_cnt": is_claim,
        "region_name": rng.choice(["01", "02", "03"], rows),
        "vehicle_type_name": rng.choice(["sedan", "truck"], rows),
        "engine_power_mean": rng.uniform(70.0, 280.0, rows),
    }
)

train_df = df.iloc[:140].copy()
valid_df = df.iloc[140:].copy()

candidate_params = {
    "catboost_freq_sev": {
        "iterations": 30,
        "reg_iterations": 30,
        "depth": 4,
        "reg_depth": 4,
        "thread_count": 1,
    },
    "oof_blend_freq_sev": {
        "base_candidates": ["baseline_freq_sev", "catboost_freq_sev"],
        "oof_folds": 3,
        "oof_group_column": "contract_number",
        "weight_grid_step": 0.5,
    },
}

if HAS_GPU:
    candidate_params["catboost_freq_sev"]["task_type"] = "GPU"
    candidate_params["catboost_freq_sev"]["devices"] = "0"

benchmark_cfg = BenchmarkConfig.from_dict(
    {
        "enabled": True,
        "selection_metric": "policy_score",
        "candidates": ["baseline_freq_sev", "catboost_freq_sev", "oof_blend_freq_sev"],
        "constraints": {
            "max_violations": 1000,
            "lr_total_min": 0.0,
            "lr_total_max": 5.0,
            "share_group1_min": 0.0,
            "share_group1_max": 1.0,
        },
        "candidate_params": candidate_params,
    }
)

result = run_model_benchmark(
    train_df=train_df,
    valid_df=valid_df,
    benchmark_config=benchmark_cfg,
    pricing_target_lr=0.7,
    pricing_alpha_grid=np.linspace(-0.6, 2.0, 15),
    pricing_beta_grid=np.linspace(1.0, 1.0, 1),
    pricing_target_band=(0.69, 0.71),
    model_max_iter=120,
    model_ridge_alpha=1.0,
)

summary = []
for item in result.results:
    pricing = item.pricing or {}
    summary.append(
        {
            "candidate": item.candidate_name,
            "status": item.status,
            "policy_score": pricing.get("policy_score"),
            "lr_total": pricing.get("lr_total"),
            "passes_constraints": item.passes_constraints,
            "elapsed_seconds": item.elapsed_seconds,
        }
    )

pd.DataFrame(summary).sort_values(["status", "policy_score"], ascending=[True, False])


,candidate,status,policy_score,lr_total,passes_constraints,elapsed_seconds
0,baseline_freq_sev,ok,-1.476079,0.039706,True,0.182444
2,oof_blend_freq_sev,ok,-1.476079,0.039706,True,3.178461
1,catboost_freq_sev,ok,-1.491725,0.043591,True,2.195813


In [8]:
print("Winner:", result.winner_name)
winner = next(item for item in result.results if item.candidate_name == result.winner_name)
print("Winner metadata:")
winner.metadata or {}


Winner: baseline_freq_sev
Winner metadata:


{}

## Step 2 - Optional real pipeline run

Run this only if the expected dataset paths from `configs/colab_gpu.json` exist in your Colab or mounted Drive workspace.


In [9]:
import subprocess

config_path = REPO_DIR / "configs" / "colab_gpu.json"
train_path = REPO_DIR / "final_dataset" / "final_dataset" / "train.csv"

print({
    "config_exists": config_path.exists(),
    "train_exists": train_path.exists(),
    "config_path": str(config_path),
    "train_path": str(train_path),
})

# Uncomment to launch the real pipeline when the dataset is available.
# subprocess.run(
#     [sys.executable, "scripts/run_pipeline.py", "--config", str(config_path)],
#     cwd=str(REPO_DIR),
#     check=True,
# )


{'config_exists': True, 'train_exists': False, 'config_path': '/content/risk-case-championship/configs/colab_gpu.json', 'train_path': '/content/risk-case-championship/final_dataset/final_dataset/train.csv'}


In [10]:
# Optional stability-check command with checkpoint/resume.
# Run from a notebook cell or terminal when you are ready for a longer job.

resume_hint = (
    "python scripts/run_stability_checks.py "
    "--config configs/colab_gpu.json "
    "--splits_config configs/experiments/stability_splits.json "
    "--candidate oof_blend_freq_sev"
)

print(resume_hint)
print("Resume form:")
print(resume_hint + " --resume_from artifacts/stability/<timestamp>/oof_blend_checkpoint")


python scripts/run_stability_checks.py --config configs/colab_gpu.json --splits_config configs/experiments/stability_splits.json --candidate oof_blend_freq_sev
Resume form:
python scripts/run_stability_checks.py --config configs/colab_gpu.json --splits_config configs/experiments/stability_splits.json --candidate oof_blend_freq_sev --resume_from artifacts/stability/<timestamp>/oof_blend_checkpoint
